In [5]:
import pandas as pd
from openpyxl import load_workbook

PATH  = r"C:\Users\Mayur Patel\Documents\GitHub\Capacity_Calculation-pipeline-optimized\data\validation\sample_version_1.xlsx"
SHEET = 0
COLORS = {"C6EFCE": "green", "FFEB9C": "yellow", "FFC7CE": "red"}
ALL_COLUMNS = False        # True -> add a _color column for every column, even uncoloured ones

df = pd.read_excel(PATH, sheet_name=SHEET)
ws = load_workbook(PATH).worksheets[SHEET]

def hexof(cell):
    f = cell.fill
    if f is None or f.patternType is None:
        return None
    s = f.start_color
    if s.type == "rgb" and s.rgb and s.rgb != "00000000":
        return s.rgb[-6:].upper()
    return None

fills = pd.DataFrame([[hexof(c) for c in row] for row in
                      ws.iter_rows(min_row=2, max_col=len(df.columns))])
fills = fills.reindex(index=range(len(df)), columns=range(len(df.columns)))

out = pd.DataFrame(index=df.index)
for i, col in enumerate(df.columns):
    out[col] = df[col]
    names = fills[i].map(lambda h: COLORS.get(h, h))     # unknown hex passes through as-is
    if ALL_COLUMNS or names.notna().any():
        out[f"{col}_color"] = names.values
df = out

df.head()

,mid_label,mid_label_new,mid_label_new_color,bosserhof_class_clean,bosserhof_class_clean.1,bosserhof_class_clean.1_color,osm_names,osm_names_color,gml_id,target_taz,target_taz_color,volume_m3
0,"['Leisure', 'Workers']",NaN,green,restaurants gastronomy,NaN,green,['Enoteca Vetrone'],None,236928,Rühen 7_472,None,1554.252341
1,['Workers'],NaN,yellow,normal office,NaN,yellow,NaN,None,535772,WOB Ehmen 2_289,None,453.347111
2,['Workers'],NaN,yellow,industrial operations production,NaN,yellow,NaN,None,477334,Gifhorn 04_416,None,727.942206
3,"['Retail_Daily', 'Workers', 'Retail_Non-Daily']","['Workers', 'Retail_Non-Daily']",red,retail small scale,NaN,green,"[""Deutsche Bank;Ernsting's family""]",None,388629,Goslar 18_565,None,17437.047953
4,['Workers'],NaN,yellow,normal office,NaN,yellow,NaN,None,81399,Bad Harzburg Schlewecke_518,None,248.752010


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1391 entries, 0 to 1390
Data columns (total 12 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   mid_label                      1391 non-null   object 
 1   mid_label_new                  365 non-null    object 
 2   mid_label_new_color            1232 non-null   object 
 3   bosserhof_class_clean          1391 non-null   object 
 4   bosserhof_class_clean.1        234 non-null    object 
 5   bosserhof_class_clean.1_color  1232 non-null   object 
 6   osm_names                      782 non-null    object 
 7   osm_names_color                50 non-null     object 
 8   gml_id                         1391 non-null   int64  
 9   target_taz                     1388 non-null   object 
 10  target_taz_color               90 non-null     object 
 11  volume_m3                      1391 non-null   float64
dtypes: float64(1), int64(1), object(10)
memory usage

In [7]:
df = df.rename(columns={
    "mid_label": "Predicted_activities",
    "mid_label_new": "Predicted_activities_mistakes",
    "mid_label_new_color": "Predicted_activities_mistakes_color",
    "bosserhof_class_clean": "Bosserhof_class_predicted",
    "bosserhof_class_clean.1": "Bosserhof_class_mistakes",
    "bosserhof_class_clean.1_color": "Bosserhof_class_mistakes_color",
    "osm_names": "osm_names",
    "gml_id": "gml_id",
    "target_taz": "target_taz",
    "volume_m3": "volume_m3"
    })

df.head()

,Predicted_activities,Predicted_activities_mistakes,Predicted_activities_mistakes_color,Bosserhof_class_predicted,Bosserhof_class_mistakes,Bosserhof_class_mistakes_color,osm_names,osm_names_color,gml_id,target_taz,target_taz_color,volume_m3
0,"['Leisure', 'Workers']",NaN,green,restaurants gastronomy,NaN,green,['Enoteca Vetrone'],None,236928,Rühen 7_472,None,1554.252341
1,['Workers'],NaN,yellow,normal office,NaN,yellow,NaN,None,535772,WOB Ehmen 2_289,None,453.347111
2,['Workers'],NaN,yellow,industrial operations production,NaN,yellow,NaN,None,477334,Gifhorn 04_416,None,727.942206
3,"['Retail_Daily', 'Workers', 'Retail_Non-Daily']","['Workers', 'Retail_Non-Daily']",red,retail small scale,NaN,green,"[""Deutsche Bank;Ernsting's family""]",None,388629,Goslar 18_565,None,17437.047953
4,['Workers'],NaN,yellow,normal office,NaN,yellow,NaN,None,81399,Bad Harzburg Schlewecke_518,None,248.752010


In [8]:
df = df[[
    "gml_id",
    "osm_names",
    "Predicted_activities",
    "Predicted_activities_mistakes",
    "Predicted_activities_mistakes_color",
    "Bosserhof_class_predicted",
    "Bosserhof_class_mistakes",
    "Bosserhof_class_mistakes_color",
    "volume_m3"
]]

df.head()

,gml_id,osm_names,Predicted_activities,Predicted_activities_mistakes,Predicted_activities_mistakes_color,Bosserhof_class_predicted,Bosserhof_class_mistakes,Bosserhof_class_mistakes_color,volume_m3
0,236928,['Enoteca Vetrone'],"['Leisure', 'Workers']",NaN,green,restaurants gastronomy,NaN,green,1554.252341
1,535772,NaN,['Workers'],NaN,yellow,normal office,NaN,yellow,453.347111
2,477334,NaN,['Workers'],NaN,yellow,industrial operations production,NaN,yellow,727.942206
3,388629,"[""Deutsche Bank;Ernsting's family""]","['Retail_Daily', 'Workers', 'Retail_Non-Daily']","['Workers', 'Retail_Non-Daily']",red,retail small scale,NaN,green,17437.047953
4,81399,NaN,['Workers'],NaN,yellow,normal office,NaN,yellow,248.752010


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1391 entries, 0 to 1390
Data columns (total 9 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   gml_id                               1391 non-null   int64  
 1   osm_names                            782 non-null    object 
 2   Predicted_activities                 1391 non-null   object 
 3   Predicted_activities_mistakes        365 non-null    object 
 4   Predicted_activities_mistakes_color  1232 non-null   object 
 5   Bosserhof_class_predicted            1391 non-null   object 
 6   Bosserhof_class_mistakes             234 non-null    object 
 7   Bosserhof_class_mistakes_color       1232 non-null   object 
 8   volume_m3                            1391 non-null   float64
dtypes: float64(1), int64(1), object(7)
memory usage: 97.9+ KB


In [10]:
df['Bosserhof_class_mistakes_color'].value_counts()

Bosserhof_class_mistakes_color
green     720
yellow    341
red       171
Name: count, dtype: int64

In [12]:
df[df['Bosserhof_class_mistakes_color'] == 'red'].head(20)

,gml_id,osm_names,Predicted_activities,Predicted_activities_mistakes,Predicted_activities_mistakes_color,Bosserhof_class_predicted,Bosserhof_class_mistakes,Bosserhof_class_mistakes_color,volume_m3
9,453160,NaN,"['Retail_Non-Daily', 'Retail_Daily']",['Retail_Non-Daily],red,retail small scale,business oriented services,red,1515.565616
10,21469,[Kunstatelier],['Workers'],"['Workers', 'Leisure']",red,normal office,entertainment culture,red,93.639860
16,175639,NaN,['Workers'],"['Workers', 'Retail_Non-Daily]",red,services,retail small scale,red,107.270188
24,296256,['Bundespolizei Fliegerstaffel Gifforn'],"['Workers', 'Retail_Daily']",['Workers'],red,services,public facilities,red,5604.546125
27,346648,NaN,"['Workers', 'Retail_Daily']","['Workers', 'School']",red,customer oriented services,school,red,389.094460
31,86341,NaN,['Workers'],NaN,green,retail,seems more like a normal office,red,79.441499
57,556592,['Bäckerei Kretzschmar'],"['Retail_Daily', 'Workers']",NaN,green,retail small scale,restaurant gastronomy,red,2377.321458
82,40531,['Krumpholz Bürosysteme'],['Workers'],NaN,green,normal office,business oriented services,red,11954.075273
97,353543,['Versöhnungskirche'],"['Leisure', 'Workers']",NaN,green,entertainment culture,public facilities,red,1339.829736
98,360068,(Psychotherapy),"['Retail_Non-Daily', 'Retail_Daily']",['Workers'],red,retail small scale,customer oriented service,red,3301.928800


In [14]:
df['Predicted_activities_mistakes_color'].value_counts()

Predicted_activities_mistakes_color
green     557
red       340
yellow    335
Name: count, dtype: int64

In [15]:
df[df['Predicted_activities_mistakes_color'] == 'red'].head(20)

,gml_id,osm_names,Predicted_activities,Predicted_activities_mistakes,Predicted_activities_mistakes_color,Bosserhof_class_predicted,Bosserhof_class_mistakes,Bosserhof_class_mistakes_color,volume_m3
3,388629,"[""Deutsche Bank;Ernsting's family""]","['Retail_Daily', 'Workers', 'Retail_Non-Daily']","['Workers', 'Retail_Non-Daily']",red,retail small scale,NaN,green,17437.047953
6,36228,"['Nazar Trockenfrüchte', ""Sara's Collection"", ...","['Retail_Daily', 'Retail_Non-Daily', 'Universi...","['Retail_Daily', 'Retail_Non-Daily', 'Universi...",red,retail small scale,NaN,green,12947.777792
8,389444,NaN,"['Workers', 'Retail_Daily']","['Workers', 'Retail_Daily', 'Retail_Non-Daily']",red,customer oriented services,NaN,green,105.044662
9,453160,NaN,"['Retail_Non-Daily', 'Retail_Daily']",['Retail_Non-Daily],red,retail small scale,business oriented services,red,1515.565616
10,21469,[Kunstatelier],['Workers'],"['Workers', 'Leisure']",red,normal office,entertainment culture,red,93.639860
16,175639,NaN,['Workers'],"['Workers', 'Retail_Non-Daily]",red,services,retail small scale,red,107.270188
17,356382,NaN,"['Workers', 'Retail_Daily']",['Workers'],red,retail small scale,NaN,green,2839.370760
23,228704,NaN,"['Retail_Non-Daily', 'Retail_Daily', 'Workers']","['Retail_Non-Daily', 'Retail_Daily', 'Workers'...",red,retail small scale,also restaurant,green,10297.225309
24,296256,['Bundespolizei Fliegerstaffel Gifforn'],"['Workers', 'Retail_Daily']",['Workers'],red,services,public facilities,red,5604.546125
27,346648,NaN,"['Workers', 'Retail_Daily']","['Workers', 'School']",red,customer oriented services,school,red,389.094460


In [16]:
COLS = ["Predicted_activities_mistakes_color", "Bosserhof_class_mistakes_color"]

df = df[df[COLS].isin(["red", "green"]).all(axis=1)].reset_index(drop=True)

print(len(df))

889


In [18]:
df['Predicted_activities_mistakes_color'].value_counts()

Predicted_activities_mistakes_color
green    550
red      339
Name: count, dtype: int64

In [19]:
df['Bosserhof_class_mistakes_color'].value_counts()

Bosserhof_class_mistakes_color
green    719
red      170
Name: count, dtype: int64

In [22]:
df[df['Bosserhof_class_mistakes_color'] == 'red']

,gml_id,osm_names,Predicted_activities,Predicted_activities_mistakes,Predicted_activities_mistakes_color,Bosserhof_class_predicted,Bosserhof_class_mistakes,Bosserhof_class_mistakes_color,volume_m3
6,453160,NaN,"['Retail_Non-Daily', 'Retail_Daily']",['Retail_Non-Daily],red,retail small scale,business oriented services,red,1515.565616
7,21469,[Kunstatelier],['Workers'],"['Workers', 'Leisure']",red,normal office,entertainment culture,red,93.639860
9,175639,NaN,['Workers'],"['Workers', 'Retail_Non-Daily]",red,services,retail small scale,red,107.270188
13,296256,['Bundespolizei Fliegerstaffel Gifforn'],"['Workers', 'Retail_Daily']",['Workers'],red,services,public facilities,red,5604.546125
15,346648,NaN,"['Workers', 'Retail_Daily']","['Workers', 'School']",red,customer oriented services,school,red,389.094460
...,...,...,...,...,...,...,...,...,...
819,199804,NaN,['Workers'],NaN,red,industrial operations production,NaN,red,30662.787271
827,356747,['Haus Leuenturm;Haus zum Stern'],"['Workers', 'Retail_Daily']","['Workers', 'Leisure']",red,retail,restaurant gastronomy,red,3297.492139
832,35606,['Gymansium Kleine Burg'],"['Retail_Daily', 'Workers']","['School', 'Workers']",red,public facilities,school,red,351.671150
845,358449,['AGT Übungsanlage;Prüfstand;Werkstatt'],"['Workers', 'Leisure']",['Workers'],red,public facilities,craft businesses,red,3018.594048


In [25]:
import geopandas as gpd

PATH = r"C:\Users\Mayur Patel\Documents\GitHub\Capacity_Calculation-pipeline-optimized\data\output\05_condensed_buildings_with_pois.gpkg"

gdf = gpd.read_file(PATH)
gdf.head()

,gml_id,volume_m3,function,label_en,osm_building_type,osm_landuse_class,osm_landuse_name,gfk_class,ALKIS_Landuse_info,osm_names,...,email,amenity,building,shop,tourism,information,tags_search,additional_information,website,geometry
0,0,931.423451,31001_2000,Buildings for business or commerce,None,residential,None,Gebäude,commercial,None,...,None,None,None,None,None,None,None,None,None,MULTIPOLYGON Z (((609554.181 5797264.172 78.94...
1,1,1275.527533,31001_2000,Buildings for business or commerce,None,residential,None,Gebäude,commercial,None,...,None,None,None,None,None,None,None,None,None,MULTIPOLYGON Z (((608098.849 5796746.146 83.91...
2,2,229.902499,31001_2000,Buildings for business or commerce,None,residential,None,Gebäude,residential,None,...,None,None,None,None,None,None,None,None,None,"MULTIPOLYGON Z (((608926.355 5797165.768 84.6,..."
3,3,84.748005,31001_2000,Buildings for business or commerce,None,residential,None,Gebäude,residential,None,...,None,None,None,None,None,None,None,None,None,MULTIPOLYGON Z (((609588.957 5797487.563 77.14...
4,4,351.716079,31001_2000,Buildings for business or commerce,None,residential,None,Gebäude,residential,None,...,None,None,None,None,None,None,None,None,None,MULTIPOLYGON Z (((608902.529 5796965.286 84.84...


In [27]:
gdf.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 574435 entries, 0 to 574434
Data columns (total 21 columns):
 #   Column                  Non-Null Count   Dtype   
---  ------                  --------------   -----   
 0   gml_id                  574435 non-null  int64   
 1   volume_m3               574435 non-null  float64 
 2   function                529997 non-null  object  
 3   label_en                529997 non-null  object  
 4   osm_building_type       56883 non-null   object  
 5   osm_landuse_class       516774 non-null  object  
 6   osm_landuse_name        46999 non-null   object  
 7   gfk_class               529997 non-null  object  
 8   ALKIS_Landuse_info      505745 non-null  object  
 9   osm_names               19871 non-null   object  
 10  alkis_address           529997 non-null  object  
 11  email                   906 non-null     object  
 12  amenity                 7863 non-null    object  
 13  building                5047 non-null    object  
 